# CNN iterativa – altura de edificações pixel a pixel (busca de arquitetura sobre o dado sintético vencedor)
Alvo denso `bldg_height` (11×11, log1p) com loss **RMSE mascarada**. Dados de treino: treino real rotacionado 4x + `smote_cgan.npz` (vencedor do SMOTE v2).
Fluxo: setup → dados e split → cGAN (com guardas) → pré-processamento → busca em 3 rodadas → contexto 11/21/33 → hold-out → patches e mapas → **teste (última célula, comentada)**.

**Regras anti-vazamento:** split `crc32` por bloco (OUTER_FOLD=0 hold-out; INNER_VAL_FOLD=1 para early stopping e seleção); normalização só do treino real;
sintético e rotações só no treino; o teste (`patch_dataset_test/`) só na última célula.

## 1. Setup e GPU

In [ ]:
# Setup: paths and shared folds (cnn/cnn_setup.py). Run this notebook from the repository root or from cnn/notebooks.
import sys, pathlib
for _p in (pathlib.Path.cwd(), pathlib.Path.cwd().parent, pathlib.Path.cwd() / "cnn"):
    if (_p / "cnn_setup.py").exists():
        sys.path.insert(0, str(_p)); break
import cnn_setup as cs

In [ ]:
# pip install "tensorflow[and-cuda]" pandas numpy mlflow matplotlib   (instala CUDA e cuDNN compatíveis)
import gc, glob, hashlib, json, os, shutil, zlib
from pathlib import Path
import numpy as np, pandas as pd, tensorflow as tf, mlflow
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
np.random.seed(SEED); tf.random.set_seed(SEED)

gpus = tf.config.list_physical_devices("GPU")
for g in gpus:
    tf.config.experimental.set_memory_growth(g, True)

info = tf.sysconfig.get_build_info()
print("TF", tf.__version__, "| CUDA", info.get("cuda_version"), "| cuDNN", info.get("cudnn_version"))
print("GPUs:", gpus)

## 2. Configuração e MLflow (experimento novo, com retomada automática)

In [ ]:
TRAIN_DIR = str(cs.TRAIN_DIR)
TEST_DIR = str(cs.TEST_DIR)                                       # lido somente na última célula (comentada)
TRAIN_PATTERN = "*.tfrecord.gz"
TEST_PATTERN = "*.tfrecord.gz"
DIST_GLOB = str(cs.DIST_GLOB)                   # distâncias teste→treino, para o teste "clean"

C_IN, C_OUT, S, NODATA = 48, 3, 33, -9999.0
PATCH_PX = 11                                                         # a cGAN do SMOTE v2 só vale para 11
K_FOLDS, OUTER_FOLD = 5, 0                                            # OUTER_FOLD: hold-out intocado até o fim
INNER_VAL_FOLD = (OUTER_FOLD + 1) % K_FOLDS                           # early stopping e seleção
BATCH, ARCH_EPOCHS, LR, DROPOUT = 128, 100, 1e-3, 0.3                 # mesmos valores da baseline do SMOTE v2
PRED_BATCH = 128

PAPER_BINS = [-np.inf, 2, 6, 15, 30, np.inf]                          # classes do artigo (RMSE macro), fechadas à direita
PAPER_NAMES = ["h_lt2", "h_2_6", "h_6_15", "h_15_30", "h_gt30"]
V2_BINS = [-0.1, 3, 6, 12, 30, np.inf]                                # classes do SMOTE v2 (só para descrever o sintético)
V2_NAMES = ["h_0_3", "h_3_6", "h_6_12", "h_12_30", "h_30p"]

SMOTE_DIR, SMOTE_FILE = str(cs.OUT / "smote_v2"), "smote_cgan.npz"
EXPECTED_N_REAL = 10339                                               # treino real com data/folds.csv (folds 0 e 1 fora); a cGAN precisa ser regerada com este treino
AUG_ROTS = (0, 1, 2, 3)                                               # rotações de 90° do treino real

CONTEXT_PX, CONTEXT_BATCH = (11, 21, 33), 64                          # experimento de contexto (sem cGAN)

RUN_TAG = "iter_v1"
OUT_DIR = str(cs.OUT / "runs_cnn_iterative")                                        # banco, artefatos e modelos desta versão
MLFLOW_URI = f"sqlite:///{OUT_DIR}/mlflow.db"
MLFLOW_EXPERIMENT = "cnn_iterative_pixelwise_arch"
MLFLOW_PORT = 5002
RESUME_RUN_ID = None                                                  # força a retomada de um run pai específico
AUTO_RESUME = True                                                    # reabre o último run incompleto com a mesma config

os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
CONFIG = {"train_dir": TRAIN_DIR, "train_pattern": TRAIN_PATTERN, "fold_method": "folds_csv_block", "patch_px": PATCH_PX,
          "k_folds": K_FOLDS, "outer_fold": OUTER_FOLD, "inner_val_fold": INNER_VAL_FOLD, "seed": SEED,
          "smote": f"{SMOTE_DIR}/{SMOTE_FILE}", "aug_rots": list(AUG_ROTS), "loss": "masked_rmse_log1p",
          "batch": BATCH, "epochs": ARCH_EPOCHS, "lr": LR, "dropout": DROPOUT, "plan": "3_rounds_v1"}
CONFIG_HASH = hashlib.md5(json.dumps(CONFIG, sort_keys=True).encode()).hexdigest()[:12]

mlflow.set_tracking_uri(MLFLOW_URI)
if mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT) is None:          # artefatos dentro de OUT_DIR
    mlflow.create_experiment(MLFLOW_EXPERIMENT, artifact_location=Path(OUT_DIR, "mlartifacts").resolve().as_uri())
mlflow.set_experiment(MLFLOW_EXPERIMENT)
if mlflow.active_run():
    mlflow.end_run(status="KILLED")                                   # run aberto por execução anterior: fica retomável

if RESUME_RUN_ID is None and AUTO_RESUME:                             # último run incompleto com a mesma configuração
    cand = mlflow.search_runs(experiment_names=[MLFLOW_EXPERIMENT], max_results=50, order_by=["attributes.start_time DESC"],
                              filter_string=f"tags.mlflow.runName = 'pipeline' and tags.config_hash = '{CONFIG_HASH}'")
    cand = cand[cand["status"] != "FINISHED"] if len(cand) else cand
    if len(cand):
        RESUME_RUN_ID = cand["run_id"].iloc[0]

if RESUME_RUN_ID:
    mlflow.start_run(run_id=RESUME_RUN_ID)
    client = mlflow.tracking.MlflowClient()
    stale = client.search_runs([mlflow.active_run().info.experiment_id],
                               filter_string=f"tags.mlflow.parentRunId = '{RESUME_RUN_ID}' and attributes.status = 'RUNNING'")
    for r in stale:
        client.set_terminated(r.info.run_id, "KILLED")                # iteração que ficou aberta quando o kernel caiu
    print("Retomando o run", RESUME_RUN_ID, f"({len(stale)} iteração(ões) interrompida(s) serão refeitas)")
else:
    mlflow.start_run(run_name="pipeline")
    print("Novo run", mlflow.active_run().info.run_id)

PARENT_RUN_ID = mlflow.active_run().info.run_id
RUN_DIR = f"{OUT_DIR}/{PARENT_RUN_ID}"                                # modelos, tabelas e figuras só desta execução
for sub in ("models_arch", "models_ctx", "figures"):
    os.makedirs(f"{RUN_DIR}/{sub}", exist_ok=True)

def out(name):
    return f"{RUN_DIR}/{name}"

mlflow.set_tags({"notebook_version": RUN_TAG, "config_hash": CONFIG_HASH, "train_dir": TRAIN_DIR,
                 "smote_variant": "rot4x_plus_cgan", "tf_version": tf.__version__, "n_gpus": str(len(gpus))})
mlflow.log_params({"seed": SEED, "patch_px": PATCH_PX, "k_folds": K_FOLDS, "outer_fold": OUTER_FOLD,
                   "inner_val_fold": INNER_VAL_FOLD, "fold_method": "folds_csv_block", "epochs_max": ARCH_EPOCHS,
                   "batch": BATCH, "lr": LR, "dropout": DROPOUT, "loss": "masked_rmse_log1p",
                   "aug_rots": str(AUG_ROTS), "smote_file": f"{SMOTE_DIR}/{SMOTE_FILE}"})

## 3. Dados: leitura (só `train`), split por bloco e checagens de vazamento
O teste não é lido aqui. Alvo denso: `bldg_height` (sem prédio = 0 m), máscara = `ndsm_mean` válido.

In [ ]:
spec = {"x": tf.io.FixedLenFeature([C_IN * S * S], tf.float32),
        "y": tf.io.FixedLenFeature([C_OUT * S * S], tf.float32),
        "height_m": tf.io.FixedLenFeature([1], tf.float32),
        "block_x": tf.io.FixedLenFeature([1], tf.float32),
        "block_y": tf.io.FixedLenFeature([1], tf.float32),
        "sample_id": tf.io.FixedLenFeature([1], tf.string),
        "city": tf.io.FixedLenFeature([1], tf.string),
        "split": tf.io.FixedLenFeature([1], tf.string)}

def make_parse(px):
    h = px // 2
    def parse(rec):
        e = tf.io.parse_single_example(rec, spec)
        x = tf.transpose(tf.reshape(e["x"], [C_IN, S, S]), [1, 2, 0])[16 - h:17 + h, 16 - h:17 + h]
        y = tf.transpose(tf.reshape(e["y"], [C_OUT, S, S]), [1, 2, 0])[16 - h:17 + h, 16 - h:17 + h]
        valid = tf.cast(y[..., 0] != NODATA, tf.float32)
        bldg_h = tf.where(y[..., 1] == NODATA, 0.0, tf.maximum(y[..., 1], 0.0))
        return (x, bldg_h, valid, e["height_m"][0], e["block_x"][0], e["block_y"][0],
                e["sample_id"][0], e["city"][0], e["split"][0])
    return parse

def subset(d, m):
    return {k: v[m] for k, v in d.items()}

def load_dense(folder, pattern, px, only_split="train", sid_whitelist=None):
    files = sorted(glob.glob(f"{folder}/{pattern}", recursive=True))
    assert files, f"nenhum arquivo em {folder}/{pattern}"
    ds = tf.data.TFRecordDataset(files, compression_type="GZIP", num_parallel_reads=tf.data.AUTOTUNE)
    ds = ds.map(make_parse(px), num_parallel_calls=tf.data.AUTOTUNE)
    if only_split is not None:
        ds = ds.filter(lambda *a: tf.equal(a[8], only_split))
    if sid_whitelist is not None:
        keys = tf.constant(sorted(sid_whitelist))
        table = tf.lookup.StaticHashTable(tf.lookup.KeyValueTensorInitializer(keys, tf.ones(len(keys), tf.int64)), 0)
        ds = ds.filter(lambda *a: table.lookup(a[6]) > 0)
    cols = [np.concatenate(c) for c in zip(*ds.batch(512).prefetch(1).as_numpy_iterator())]
    d = dict(zip(["x", "h_map", "mask_px", "h", "bx", "by", "sid", "city", "split"], cols))
    for k in ("sid", "city", "split"):
        d[k] = np.array([s.decode() for s in d[k]])
    _, keep = np.unique(d["sid"], return_index=True)                  # sample_id repetido entre arquivos
    return {k: v[np.sort(keep)] for k, v in d.items()}

fold_of = cs.fold_of            # folds from data/folds.csv, shared with the classical models

ds_train = load_dense(TRAIN_DIR, TRAIN_PATTERN, PATCH_PX)
fold = fold_of(ds_train["city"], ds_train["bx"], ds_train["by"], K_FOLDS)
assert set(np.unique(fold)) == set(range(K_FOLDS)), f"folds encontrados: {sorted(set(fold))}"
assert INNER_VAL_FOLD != OUTER_FOLD

tr_split = subset(ds_train, ~np.isin(fold, [OUTER_FOLD, INNER_VAL_FOLD]))
va = subset(ds_train, fold == INNER_VAL_FOLD)                         # validação interna: early stopping e seleção
ho = subset(ds_train, fold == OUTER_FOLD)                             # hold-out: só na seção 11
SID_TR, SID_VA = tr_split["sid"].copy(), va["sid"].copy()
n_records = len(ds_train["h"])
ds_train = None
gc.collect()
print("treino real:", len(tr_split["h"]), "| validação interna:", len(va["h"]), "| hold-out:", len(ho["h"]))
print("por cidade (treino):", pd.Series(tr_split["city"]).value_counts().to_dict())
mlflow.log_param("n_records_train_files", n_records)

In [ ]:
# checagens de vazamento
sets = {"treino": set(tr_split["sid"]), "val_interna": set(va["sid"]), "holdout": set(ho["sid"])}
assert not (sets["treino"] & sets["val_interna"]) and not (sets["treino"] & sets["holdout"]) and not (sets["val_interna"] & sets["holdout"])
assert (np.isin(fold_of(tr_split["city"], tr_split["bx"], tr_split["by"], K_FOLDS), [OUTER_FOLD, INNER_VAL_FOLD]) == False).all()
assert (fold_of(va["city"], va["bx"], va["by"], K_FOLDS) == INNER_VAL_FOLD).all()
assert (fold_of(ho["city"], ho["bx"], ho["by"], K_FOLDS) == OUTER_FOLD).all()
assert set(tr_split["split"]) == set(va["split"]) == set(ho["split"]) == {"train"}
assert len(tr_split["h"]) == EXPECTED_N_REAL, (
    f"treino real = {len(tr_split['h'])}, esperado {EXPECTED_N_REAL}: o dataset mudou desde a geração da cGAN; regenere o sintético")
print("OK: conjuntos disjuntos, folds coerentes, só split == train, tamanho do treino igual ao da geração da cGAN")

## 4. Sintético vencedor do SMOTE v2: cGAN conjunta (com guardas)
Se as estatísticas de normalização não baterem com as do treino atual, a cGAN foi gerada com outro treino e o notebook para.

In [ ]:
def channel_stats(X, chunk=2048):
    n = np.zeros(X.shape[-1]); s = np.zeros(X.shape[-1]); s2 = np.zeros(X.shape[-1])
    for i in range(0, len(X), chunk):
        xb = X[i:i + chunk].astype("float64")
        v = xb != NODATA
        xb = np.where(v, xb, 0.0)
        n += v.sum((0, 1, 2)); s += xb.sum((0, 1, 2)); s2 += (xb ** 2).sum((0, 1, 2))
    mean = s / n.clip(1)
    var = s2 / n.clip(1) - mean ** 2
    return mean.astype("float32"), np.sqrt(var.clip(0)).clip(1e-6).astype("float32")

MEAN, STD = channel_stats(tr_split["x"])                              # só treino real

d_gan = np.load(os.path.join(SMOTE_DIR, SMOTE_FILE))
assert int(d_gan["patch_px"]) == PATCH_PX, f"cGAN gerada com patch_px={int(d_gan['patch_px'])}"
assert np.allclose(MEAN, d_gan["mean"], rtol=1e-6, atol=1e-7) and np.allclose(STD, d_gan["std"], rtol=1e-6, atol=1e-7), (
    "MEAN/STD diferem dos da cGAN: ela foi gerada com outro treino (risco de vazamento); regenere o sintético")

X_syn = d_gan["X_syn"].astype("float32")
Y_syn = np.stack([d_gan["y_map_syn"], d_gan["mask_syn"]], -1).astype("float32")     # log1p(m) e máscara
h_syn = d_gan["h_syn"].astype("float32")
assert X_syn.shape[1:] == (PATCH_PX, PATCH_PX, C_IN + 8) and Y_syn.shape[1:3] == (PATCH_PX, PATCH_PX)
assert len(X_syn) == len(Y_syn) == len(h_syn) and np.isfinite(X_syn).all() and np.isfinite(Y_syn).all()

syn_counts = dict(zip(V2_NAMES, np.bincount(np.digitize(h_syn, V2_BINS[1:-1], right=True), minlength=5).tolist()))
print("cGAN:", len(X_syn), "amostras | altura escalar", f"{h_syn.min():.1f}-{h_syn.max():.1f} m | por faixa:", syn_counts)
mlflow.log_params({"n_synthetic": len(X_syn), "n_real_train": len(tr_split["h"])})

## 5. Pré-processamento e plano de treino (real rotacionado 4x + cGAN)
Rotações geradas a cada batch, coerentes em `X` e `Y` (sem materializar 4x). Validação e hold-out sem aumento. Libera os arrays brutos.

In [ ]:
VALID_CH = [s * 12 + b for s in range(4) for b in (0, 2)]            # VV e B2 de cada estação

def prep(X, mean, std, chunk=2048):
    n, c = len(X), X.shape[3]
    arr = np.empty((n, X.shape[1], X.shape[2], c + len(VALID_CH)), dtype="float32")
    for s in range(0, n, chunk):
        xb = X[s:s + chunk]
        valid = xb != NODATA
        e = s + len(xb)
        arr[s:e, ..., :c] = np.where(valid, (xb - mean) / std, 0.0)
        arr[s:e, ..., c:] = valid[..., VALID_CH]
    return arr

def pack(h_map, mask):                                                # alvo (log1p) e máscara no mesmo tensor
    return np.stack([np.log1p(h_map), mask], -1).astype("float32")

def make_plan(n_real, n_syn, rots=AUG_ROTS):                          # linhas: (segmento, índice, rotação de 90°)
    parts = [np.stack([np.zeros(n_real, np.int64), np.arange(n_real), np.full(n_real, k)], 1) for k in rots]
    if n_syn:
        parts.append(np.stack([np.ones(n_syn, np.int64), np.arange(n_syn), np.zeros(n_syn, np.int64)], 1))
    return np.concatenate(parts).astype(np.int64)

X_real, Y_real = prep(tr_split["x"], MEAN, STD), pack(tr_split["h_map"], tr_split["mask_px"])
X_va, Y_va = prep(va["x"], MEAN, STD), pack(va["h_map"], va["mask_px"])
X_ho, Y_ho = prep(ho["x"], MEAN, STD), pack(ho["h_map"], ho["mask_px"])
h_map_va, mask_va = va["h_map"].astype("float32"), va["mask_px"].astype("float32")
h_map_ho, mask_ho = ho["h_map"].astype("float32"), ho["mask_px"].astype("float32")

for d in (tr_split, va, ho):
    d.pop("x", None)
gc.collect()

SEGMENTS_TR = [(X_real, Y_real), (X_syn, Y_syn)]
PLAN_TR = make_plan(len(X_real), len(X_syn))
PLAN_VA = np.stack([np.zeros(len(X_va), np.int64), np.arange(len(X_va)), np.zeros(len(X_va), np.int64)], 1)

mlflow.log_params({"n_train_total": len(PLAN_TR), "n_val": len(X_va), "n_holdout": len(X_ho), "n_channels": X_real.shape[-1]})
print("amostras de treino por época:", len(PLAN_TR), f"| X_real {X_real.nbytes / 2 ** 30:.2f} GiB + X_syn {X_syn.nbytes / 2 ** 30:.2f} GiB")
print("pixels válidos treino:", f"{Y_real[..., 1].mean() * 100:.1f}%", "| pixels com prédio:", f"{(Y_real[..., 0] > 0).mean() * 100:.1f}%")

## 6. Loss, métricas, dataset e modelos
Loss de todos os modelos: RMSE mascarado em log1p(m). MAE entra como métrica (por época em log1p e, na avaliação, em metros). Métrica de seleção: RMSE macro (média dos RMSE das 5 classes do artigo) em metros, nos pixels válidos.

In [ ]:
def masked_rmse_loss(y_pack, y_pred):                                 # y_pack[..., 0] = alvo log1p, y_pack[..., 1] = máscara
    y, m = y_pack[..., 0], y_pack[..., 1]
    sq = tf.square(y - tf.squeeze(y_pred, -1))
    return tf.sqrt(tf.reduce_sum(sq * m) / (tf.reduce_sum(m) + 1e-8) + 1e-8)

def masked_rmse_mix_loss(y_pack, y_pred):                             # 0.5 RMSE (todos) + 0.5 RMSE (pixels com prédio)
    y, m = y_pack[..., 0], y_pack[..., 1]
    sq = tf.square(y - tf.squeeze(y_pred, -1))
    mb = m * tf.cast(y > 0.0, m.dtype)
    r_all = tf.sqrt(tf.reduce_sum(sq * m) / (tf.reduce_sum(m) + 1e-8) + 1e-8)
    r_bld = tf.sqrt(tf.reduce_sum(sq * mb) / (tf.reduce_sum(mb) + 1e-8) + 1e-8)
    return 0.5 * r_all + 0.5 * r_bld

def masked_mae_log(y_pack, y_pred):
    y, m = y_pack[..., 0], y_pack[..., 1]
    return tf.reduce_sum(tf.abs(y - tf.squeeze(y_pred, -1)) * m) / (tf.reduce_sum(m) + 1e-8)

def make_ds_dense(segments, plan, batch, training, seed=SEED):       # rotaciona por batch; sem cópia dos dados
    n, rng = len(plan), np.random.default_rng(seed)
    X0, Y0 = segments[0]
    def gen():
        order = rng.permutation(n) if training else np.arange(n)
        for s in range(0, n, batch):
            p = plan[order[s:s + batch]]
            xs, ys = [], []
            for sg in np.unique(p[:, 0]):
                for rt in np.unique(p[p[:, 0] == sg, 2]):
                    sel = p[(p[:, 0] == sg) & (p[:, 2] == rt), 1]
                    xb, yb = segments[sg][0][sel], segments[sg][1][sel]
                    if rt:
                        xb, yb = np.rot90(xb, rt, axes=(1, 2)), np.rot90(yb, rt, axes=(1, 2))
                    xs.append(np.ascontiguousarray(xb)); ys.append(np.ascontiguousarray(yb))
            yield np.concatenate(xs), np.concatenate(ys)
    sig = (tf.TensorSpec((None, *X0.shape[1:]), tf.float32), tf.TensorSpec((None, *Y0.shape[1:]), tf.float32))
    return tf.data.Dataset.from_generator(gen, output_signature=sig).prefetch(tf.data.AUTOTUNE)

def predict_map_m(m, X):                                              # (N, H, W) em metros
    p = np.concatenate([m.predict_on_batch(X[s:s + PRED_BATCH]) for s in range(0, len(X), PRED_BATCH)], axis=0)
    return np.clip(np.expm1(np.asarray(p)[..., 0]), 0, None)

def _m1d(y, p):
    err = p - y
    return {"MAE": float(np.abs(err).mean()), "RMSE": float(np.sqrt((err ** 2).mean())), "bias": float(err.mean()),
            "R2": float(1.0 - (err ** 2).sum() / (((y - y.mean()) ** 2).sum() + 1e-8))}

def dense_metrics(h_true, h_pred, mask):
    valid = mask > 0.5
    y, p = h_true[valid].astype("float64"), h_pred[valid].astype("float64")
    bld = y > 0
    c = h_true.shape[1] // 2
    m_all = _m1d(y, p)
    m_bld = _m1d(y[bld], p[bld]) if bld.any() else {k: float("nan") for k in m_all}
    m_ctr = _m1d(h_true[:, c, c].astype("float64"), h_pred[:, c, c].astype("float64"))
    idx = np.digitize(y, PAPER_BINS[1:-1], right=True)
    cls_rmse, cls_mae = {}, {}
    for k, name in enumerate(PAPER_NAMES):
        s = idx == k
        if s.any():
            e = p[s] - y[s]
            cls_rmse[f"RMSE_{name}"] = float(np.sqrt((e ** 2).mean()))
            cls_mae[f"MAE_{name}"] = float(np.abs(e).mean())
    return {"macro_RMSE_px": float(np.mean(list(cls_rmse.values()))), "RMSE_px": m_all["RMSE"], "MAE_px": m_all["MAE"],
            "bias_px": m_all["bias"], "R2_px": m_all["R2"], "RMSE_bldg_px": m_bld["RMSE"], "MAE_bldg_px": m_bld["MAE"],
            "RMSE_center_px": m_ctr["RMSE"], **cls_rmse, **cls_mae}

def crop_center(a, n):
    c, h = a.shape[1] // 2, n // 2
    return a[:, c - h:c + h + 1, c - h:c + h + 1]

In [ ]:
register = getattr(getattr(keras, "saving", None), "register_keras_serializable", None) or keras.utils.register_keras_serializable

@register(package="cnn_iter")
class SeasonSplit(layers.Layer):                                      # (B,h,w,56) -> (B*4,h,w,14): 12 bandas + 2 validades por estação
    def call(self, x):
        h, w = x.shape[1], x.shape[2]
        bands = tf.reshape(x[..., :48], (-1, h, w, 4, 12))
        valid = tf.reshape(x[..., 48:], (-1, h, w, 4, 2))
        s = tf.transpose(tf.concat([bands, valid], -1), (0, 3, 1, 2, 4))
        return tf.reshape(s, (-1, h, w, 14))

    def compute_output_shape(self, input_shape):
        return (None, input_shape[1], input_shape[2], 14)

@register(package="cnn_iter")
class SeasonFuse(layers.Layer):                                       # atenção entre estações por pixel (softmax) + soma ponderada
    def __init__(self, n_seasons=4, **kwargs):
        super().__init__(**kwargs)
        self.n_seasons = n_seasons

    def call(self, inputs):
        f, a = inputs
        h, w, c = f.shape[1], f.shape[2], f.shape[3]
        f = tf.reshape(f, (-1, self.n_seasons, h, w, c))
        a = tf.reshape(a, (-1, self.n_seasons, h, w, 1))
        return tf.reduce_sum(f * tf.nn.softmax(a, axis=1), axis=1)

    def compute_output_shape(self, input_shape):
        return (None, *input_shape[0][1:])

    def get_config(self):
        return {**super().get_config(), "n_seasons": self.n_seasons}

def conv_block(x, f):
    x = layers.Conv2D(f, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    return layers.ReLU()(x)

def se_block(x, r=8):                                                 # atenção por canal (Squeeze-and-Excitation)
    c = x.shape[-1]
    s = layers.GlobalAveragePooling2D(keepdims=True)(x)
    s = layers.Dense(max(c // r, 4), activation="relu")(s)
    s = layers.Dense(c, activation="sigmoid")(s)
    return layers.Multiply()([x, s])

def build_dense_unet(input_shape, filters, convs=2, se=False, tatt=False, dropout=DROPOUT):
    h, w, ch = input_shape
    depth = len(filters) - 1
    p = (-h) % (2 ** depth)                                           # padding para dividir por 2**depth
    inp = layers.Input(input_shape)
    if tatt:                                                          # stem compartilhado por estação + atenção temporal
        assert ch == C_IN + 8
        f = conv_block(SeasonSplit()(inp), filters[0])
        f = conv_block(f, filters[0])
        x = SeasonFuse(4)([f, layers.Conv2D(1, 1)(f)])
    else:
        x = inp
    if p:
        x = layers.ZeroPadding2D(((0, p), (0, p)))(x)
    skips = []
    for f in filters[:-1]:                                            # encoder
        for _ in range(convs):
            x = conv_block(x, f)
        if se:
            x = se_block(x)
        skips.append(x)
        x = layers.MaxPool2D()(x)
    x = conv_block(x, filters[-1])                                    # gargalo
    if se:
        x = se_block(x)
    x = layers.Dropout(dropout)(x)
    for f, sk in zip(filters[-2::-1], skips[::-1]):                   # decoder com skip connections
        x = layers.UpSampling2D()(x)
        x = layers.Concatenate()([x, sk])
        for _ in range(convs):
            x = conv_block(x, f)
        if se:
            x = se_block(x)
    if p:
        x = layers.Cropping2D(((0, p), (0, p)))(x)
    return keras.Model(inp, layers.Conv2D(1, 1, activation="relu")(x))      # log1p(altura) >= 0 por pixel

def spec_of(name, filters, convs=2, se=False, tatt=False, loss="rmse"):
    return {"name": name, "filters": filters, "convs": convs, "se": se, "tatt": tatt, "loss": loss}

def build_from_spec(spec, input_shape):
    return build_dense_unet(input_shape, spec["filters"], spec["convs"], spec["se"], spec["tatt"])

In [ ]:
def free_gpu():
    keras.backend.clear_session()
    gc.collect()

def reset_gpu_peak():
    try:
        tf.config.experimental.reset_memory_stats("GPU:0")
    except Exception:
        pass

def gpu_peak_mb():
    try:
        return {"gpu_peak_mb": tf.config.experimental.get_memory_info("GPU:0")["peak"] / 2 ** 20}
    except Exception:
        return {}

def gpu_in_use():
    try:
        return f"{tf.config.experimental.get_memory_info('GPU:0')['current'] / 2 ** 20:.0f} MB"
    except Exception:
        return "n/d"

def host_rss_mb():
    try:
        import psutil
        return psutil.Process().memory_info().rss / 2 ** 20
    except Exception:
        return float("nan")

def log_resources():
    mlflow.log_metrics(gpu_peak_mb())
    if not np.isnan(host_rss_mb()):
        mlflow.log_metric("host_rss_mb", host_rss_mb())

def mlflow_epoch_cb():
    return keras.callbacks.LambdaCallback(
        on_epoch_end=lambda ep, logs: mlflow.log_metrics({k: float(v) for k, v in logs.items()}, step=ep))

def train_callbacks():
    return [keras.callbacks.EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True),
            keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=6, min_lr=1e-5),
            mlflow_epoch_cb()]

# --- retomada: o MLflow é a fonte da verdade sobre o que já terminou
def finished_children(stage):
    run = mlflow.active_run().info
    return mlflow.search_runs(
        experiment_ids=[run.experiment_id], order_by=["attributes.start_time DESC"],
        filter_string=f"tags.mlflow.parentRunId = '{run.run_id}' and tags.stage = '{stage}' and attributes.status = 'FINISHED'")

def collect_rows(stage, make_row):
    rows = [make_row(q) for _, q in finished_children(stage).iterrows()]
    return pd.DataFrame([r for r in rows if r is not None])

def run_row(q):
    key = q["tags.mlflow.runName"]
    path = out(f"{'models_arch' if q['tags.stage'] == 'arch_search' else 'models_ctx'}/{key}.keras")
    if not os.path.exists(path):
        return None                                                   # sem o modelo no disco: será refeito
    return {"key": key, **{c.replace("params.", ""): q[c] for c in q.index if c.startswith("params.")},
            **{c.replace("metrics.", ""): q[c] for c in q.index if c.startswith("metrics.val_")},
            "epochs": q.get("metrics.epochs_run"), "model_path": path}

## 7. Verificação rápida das arquiteturas
Constrói cada variante, faz um forward pass e testa salvar/recarregar (camadas customizadas) antes de gastar tempo treinando.

In [ ]:
R1_SPECS = [spec_of("u32_c2", [32, 64, 128]),                         # baseline: U-Net compacta do SMOTE v2
            spec_of("u48_c2", [48, 96, 192]),
            spec_of("u64_c2", [64, 128, 256]),
            spec_of("u32_c3", [32, 64, 128], convs=3),
            spec_of("u32_d3_c2", [32, 64, 128, 256])]

smoke = R1_SPECS + [{**R1_SPECS[0], "name": "smoke_se", "se": True}, {**R1_SPECS[0], "name": "smoke_tatt", "tatt": True},
                    {**R1_SPECS[0], "name": "smoke_se_tatt", "se": True, "tatt": True}]
dummy = np.random.default_rng(0).normal(size=(2, PATCH_PX, PATCH_PX, C_IN + 8)).astype("float32")
for s in smoke:
    m = build_from_spec(s, (PATCH_PX, PATCH_PX, C_IN + 8))
    o = m(dummy, training=False)
    assert tuple(o.shape) == (2, PATCH_PX, PATCH_PX, 1), (s["name"], o.shape)
    print(f"{s['name']:<14} saída {tuple(o.shape)} | {m.count_params():>9,} parâmetros")
    if s["name"] == "smoke_se_tatt":                                  # ida e volta com camadas customizadas
        m.save(out("figures/_smoke.keras"))
        m2 = keras.models.load_model(out("figures/_smoke.keras"), compile=False)
        assert np.allclose(m(dummy, training=False), m2(dummy, training=False), atol=1e-5)
        os.remove(out("figures/_smoke.keras"))
        print("   salvar/recarregar OK")
    del m
free_gpu()

## 8. Busca iterativa de arquitetura (3 rodadas, retomável)
Rodada 1: largura, profundidade e convoluções. Rodada 2: atenção (SE, atenção temporal por estação, ambas) sobre a melhor. Rodada 3: loss mista RMSE (todos + prédios) sobre a melhor. Seleção: menor `val_macro_RMSE_px`, desempate `val_RMSE_bldg_px`.

In [ ]:
import ctypes

assert "X_real" in globals() and "PLAN_TR" in globals(), "Execute antes as seções 2 a 7 (Run All Above): os dados vivem na memória"

WEIGHTS_DIR = "cnn_iterative_experimentation"
os.makedirs(WEIGHTS_DIR, exist_ok=True)

try:
    _LIBC = ctypes.CDLL("libc.so.6")
except Exception:
    _LIBC = None

def _release_all_memory():
    keras.backend.clear_session()
    for _ in range(3):
        gc.collect()
    if _LIBC is not None:
        try:
            _LIBC.malloc_trim(0)
        except Exception:
            pass

REF_V2 = {"RMSE_px": 10.7251, "MAE_px": 5.8732, "RMSE_bldg_px": 12.7268, "R2_px": 0.5958, "bias_px": -1.9399}   # SMOTE v2, rot4x+cGAN

def train_candidate(round_id, spec):
    key = f"{round_id}_{spec['name']}"
    m = h = cb = ds_t = ds_v = r = pred_va = None
    try:
        _release_all_memory()
        reset_gpu_peak()
        m = build_from_spec(spec, X_real.shape[1:])
        m.compile(optimizer=keras.optimizers.Adam(LR), metrics=[masked_mae_log],
                  loss=masked_rmse_mix_loss if spec["loss"] == "mix" else masked_rmse_loss)
        ckpt_tmp = os.path.join(WEIGHTS_DIR, f"{key}_partial.keras")
        if os.path.exists(ckpt_tmp):
            m.load_weights(ckpt_tmp)
            print(f"   pesos parciais carregados de {ckpt_tmp}")
        n_params = m.count_params()
        cb = list(train_callbacks()) + [keras.callbacks.ModelCheckpoint(ckpt_tmp, monitor="val_loss", save_best_only=True)]
        ds_t = make_ds_dense(SEGMENTS_TR, PLAN_TR, BATCH, True)
        ds_v = make_ds_dense([(X_va, Y_va)], PLAN_VA, BATCH, False)

        with mlflow.start_run(run_name=key, nested=True):
            mlflow.set_tags({"stage": "arch_search", "round": round_id})
            mlflow.log_params({"round": round_id, "name": spec["name"], "filters": str(spec["filters"]), "convs": spec["convs"],
                               "se": int(spec["se"]), "tatt": int(spec["tatt"]), "loss": spec["loss"], "n_params": n_params,
                               "lr": LR, "batch": BATCH, "dropout": DROPOUT})
            h = m.fit(ds_t, validation_data=ds_v, epochs=ARCH_EPOCHS, callbacks=cb, verbose=0)
            epochs_run = len(h.history["loss"])
            pred_va = predict_map_m(m, X_va)
            r = dense_metrics(h_map_va, pred_va, mask_va)
            mlflow.log_metrics({f"val_{k}": float(v) for k, v in r.items()})
            mlflow.log_metric("epochs_run", epochs_run)
            log_resources()

            os.makedirs(WEIGHTS_DIR, exist_ok=True)
            weights_path = os.path.join(WEIGHTS_DIR, f"{key}.weights.h5")
            model_path = out(f"models_arch/{key}.keras")
            m.save_weights(weights_path)
            m.save(model_path)
            mlflow.log_artifact(weights_path, artifact_path="weights")
            if os.path.exists(ckpt_tmp):
                os.remove(ckpt_tmp)

        print(f"{key:<24} params = {n_params:>9,} | macro RMSE = {r['macro_RMSE_px']:.3f} m | RMSE_px = {r['RMSE_px']:.3f} m | "
              f"RMSE_bldg = {r['RMSE_bldg_px']:.3f} m | MAE_px = {r['MAE_px']:.3f} m | épocas = {epochs_run}")
        print("   RMSE por classe:", {k.replace("RMSE_", ""): round(v, 2) for k, v in r.items() if k.startswith("RMSE_h_")})
    except tf.errors.ResourceExhaustedError:                          # o run fica FAILED e é refeito na retomada
        print(f"{key}: sem memória na GPU, pulado")
    finally:
        if h is not None and hasattr(h, "history"):
            h.history.clear()
        del m, h, cb, ds_t, ds_v, r, pred_va
        _release_all_memory()
        print(f"   RAM do processo: {host_rss_mb():.0f} MB | GPU em uso: {gpu_in_use()}")

def run_round(round_id, specs):
    done = set(collect_rows("arch_search", run_row).get("key", []))
    todo = [s for s in specs if f"{round_id}_{s['name']}" not in done]
    print(f"{round_id}: {len(specs) - len(todo)} de {len(specs)} já concluídos; restam {len(todo)}")
    for s in todo:
        train_candidate(round_id, s)
    return collect_rows("arch_search", run_row).drop_duplicates("key")

def pick_best(rows, specs):
    rows = rows[rows["name"].isin(specs)].sort_values(["val_macro_RMSE_px", "val_RMSE_bldg_px"]).reset_index(drop=True)
    return specs[rows["name"][0]], rows.iloc[0]

SHOW = ["key", "n_params", "val_macro_RMSE_px", "val_RMSE_px", "val_RMSE_bldg_px", "val_MAE_px", "val_MAE_bldg_px", "val_R2_px", "val_bias_px"]
SPECS = {s["name"]: s for s in R1_SPECS}

### 8.1 Rodada 1: largura, profundidade e convoluções por nível

In [ ]:
rows = run_round("r1", R1_SPECS)
best1, row1 = pick_best(rows[rows["round"] == "r1"], SPECS)
base_row = rows[rows["key"] == "r1_u32_c2"]
if len(base_row):
    b = base_row.iloc[0]
    print("\nbaseline vs SMOTE v2 (rot4x+cGAN):", {k: (round(float(b[f"val_{k}"]), 3), v) for k, v in REF_V2.items()})
print("melhor da rodada 1:", best1["name"])
rows[rows["round"] == "r1"].sort_values("val_macro_RMSE_px")[SHOW].round(4)

### 8.2 Rodada 2: atenção sobre a melhor da rodada 1

In [ ]:
R2_SPECS = [{**best1, "name": best1["name"] + "+se", "se": True},
            {**best1, "name": best1["name"] + "+tatt", "tatt": True},
            {**best1, "name": best1["name"] + "+se+tatt", "se": True, "tatt": True}]
SPECS.update({s["name"]: s for s in R2_SPECS})

rows = run_round("r2", R2_SPECS)
best2, row2 = pick_best(rows[rows["round"].isin(["r1", "r2"])], SPECS)
print("melhor até a rodada 2:", best2["name"])
rows[rows["round"].isin(["r1", "r2"])].sort_values("val_macro_RMSE_px")[SHOW].round(4)

### 8.3 Rodada 3: loss mista RMSE (todos os pixels + pixels com prédio)

In [ ]:
results = rows.sort_values(["val_macro_RMSE_px", "val_RMSE_bldg_px"]).reset_index(drop=True)
BEST_SPEC, best_row = pick_best(results, SPECS)
BEST_PATH = best_row["model_path"]
print(f"MELHOR MODELO: {best_row['key']} | macro RMSE = {best_row['val_macro_RMSE_px']:.3f} m | "
      f"RMSE_px = {best_row['val_RMSE_px']:.3f} m | RMSE_bldg = {best_row['val_RMSE_bldg_px']:.3f} m")
results[SHOW].round(4)

In [ ]:
#R3_SPECS = [{**best2, "name": best2["name"] + "+mix", "loss": "mix"}]
#SPECS.update({s["name"]: s for s in R3_SPECS})

#rows = run_round("r3", R3_SPECS)
#results = rows.sort_values(["val_macro_RMSE_px", "val_RMSE_bldg_px"]).reset_index(drop=True)
#BEST_SPEC, best_row = pick_best(results, SPECS)
#BEST_PATH = best_row["model_path"]
#print(f"MELHOR MODELO: {best_row['key']} | macro RMSE = {best_row['val_macro_RMSE_px']:.3f} m | "
#      f"RMSE_px = {best_row['val_RMSE_px']:.3f} m | RMSE_bldg = {best_row['val_RMSE_bldg_px']:.3f} m")
#results[SHOW].round(4)

## 9. Modelo selecionado: tabela completa e registro no MLflow

In [ ]:
results.to_csv(out("arch_search_results.csv"), index=False)
mlflow.log_artifact(out("arch_search_results.csv"))
mlflow.set_tags({"best_key": best_row["key"], "best_filters": str(BEST_SPEC["filters"]), "best_convs": str(BEST_SPEC["convs"]),
                 "best_se": str(BEST_SPEC["se"]), "best_tatt": str(BEST_SPEC["tatt"]), "best_loss": BEST_SPEC["loss"]})
mlflow.log_metrics({"best_val_macro_RMSE_px": float(best_row["val_macro_RMSE_px"]), "best_val_RMSE_px": float(best_row["val_RMSE_px"]),
                    "best_val_RMSE_bldg_px": float(best_row["val_RMSE_bldg_px"]), "best_val_MAE_px": float(best_row["val_MAE_px"])})
cols = [c for c in results.columns if c.startswith("val_RMSE_h_") or c.startswith("val_MAE_h_")]
results[["key"] + cols].round(3)

## 10. Experimento de contexto: melhor arquitetura em 11, 21 e 33 px (só treino real rotacionado, sem cGAN)
Comparação justa: todos avaliados no mesmo recorte central 11×11 da validação interna. Informativo: o restante do notebook segue em 11 px. Não toca no hold-out; reaproveita os mesmos `sample_id` de treino e validação.

In [ ]:
assert "BEST_SPEC" in globals(), "Execute a seção 8 antes"

def run_context(px):
    key = f"ctx_px{px}"
    free_gpu(); reset_gpu_peak()
    wl = set(SID_TR) | set(SID_VA)
    d = load_dense(TRAIN_DIR, TRAIN_PATTERN, px, sid_whitelist=wl)
    tr_c, va_c = subset(d, np.isin(d["sid"], SID_TR)), subset(d, np.isin(d["sid"], SID_VA))
    d = None
    mean, std = channel_stats(tr_c["x"])
    Xt, Yt = prep(tr_c["x"], mean, std), pack(tr_c["h_map"], tr_c["mask_px"])
    Xv, Yv = prep(va_c["x"], mean, std), pack(va_c["h_map"], va_c["mask_px"])
    hv, mv = va_c["h_map"].astype("float32"), va_c["mask_px"].astype("float32")
    tr_c = va_c = None
    gc.collect()
    plan_t = make_plan(len(Xt), 0)
    plan_v = np.stack([np.zeros(len(Xv), np.int64), np.arange(len(Xv)), np.zeros(len(Xv), np.int64)], 1)
    m = h = cb = ds_t = ds_v = None
    try:
        m = build_from_spec(BEST_SPEC, Xt.shape[1:])
        m.compile(optimizer=keras.optimizers.Adam(LR), metrics=[masked_mae_log],
                  loss=masked_rmse_mix_loss if BEST_SPEC["loss"] == "mix" else masked_rmse_loss)
        cb = train_callbacks()
        ds_t = make_ds_dense([(Xt, Yt)], plan_t, CONTEXT_BATCH, True)
        ds_v = make_ds_dense([(Xv, Yv)], plan_v, CONTEXT_BATCH, False)
        with mlflow.start_run(run_name=key, nested=True):
            mlflow.set_tags({"stage": "context"})
            mlflow.log_params({"px": px, "name": BEST_SPEC["name"], "train_set": "real_rot4x_sem_cgan", "batch": CONTEXT_BATCH,
                               "n_params": m.count_params(), "eval_crop": 11})
            h = m.fit(ds_t, validation_data=ds_v, epochs=ARCH_EPOCHS, callbacks=cb, verbose=0)
            pm = predict_map_m(m, Xv)
            r = dense_metrics(crop_center(hv, 11), crop_center(pm, 11), crop_center(mv, 11))     # mesmo suporte para todos
            full = dense_metrics(hv, pm, mv)
            mlflow.log_metrics({f"val_{k}": float(v) for k, v in r.items()})
            mlflow.log_metrics({"val_full_RMSE_px": full["RMSE_px"], "val_full_RMSE_bldg_px": full["RMSE_bldg_px"]})
            mlflow.log_metric("epochs_run", len(h.history["loss"]))
            log_resources()
            m.save(out(f"models_ctx/{key}.keras"))
        print(f"{key}: macro RMSE (11x11 central) = {r['macro_RMSE_px']:.3f} m | RMSE_px = {r['RMSE_px']:.3f} m | "
              f"RMSE_bldg = {r['RMSE_bldg_px']:.3f} m | MAE_px = {r['MAE_px']:.3f} m | RMSE_px mapa completo = {full['RMSE_px']:.3f} m")
    except tf.errors.ResourceExhaustedError:
        print(f"{key}: sem memória; reduza CONTEXT_BATCH")
    finally:
        del m, h, cb, ds_t, ds_v, Xt, Yt, Xv, Yv, hv, mv
        free_gpu()
        print(f"   RAM do processo: {host_rss_mb():.0f} MB | GPU em uso: {gpu_in_use()}")

done_ctx = set(collect_rows("context", run_row).get("key", []))
for px in CONTEXT_PX:
    if f"ctx_px{px}" in done_ctx:
        continue
    run_context(px)

ctx = collect_rows("context", run_row).drop_duplicates("key").sort_values("px").reset_index(drop=True)
ctx_cols = ["key", "val_macro_RMSE_px", "val_RMSE_px", "val_RMSE_bldg_px", "val_MAE_px", "val_MAE_bldg_px", "val_R2_px", "val_full_RMSE_px"]
ctx[[c for c in ctx_cols if c in ctx.columns]].round(4)

## 11. Hold-out (fold externo) com o modelo selecionado
Primeiro uso do hold-out: a arquitetura já está congelada. Gera as predições do fold (pixel central, em metros).

In [ ]:
assert "BEST_PATH" in globals(), "Execute a seção 8 antes"
free_gpu()
model = keras.models.load_model(BEST_PATH, compile=False)
p_ho = predict_map_m(model, X_ho)
ho_m = dense_metrics(h_map_ho, p_ho, mask_ho)
mlflow.log_metrics({f"ho_{k}": float(v) for k, v in ho_m.items()})

oof = pd.DataFrame({"sample_id": ho["sid"], "fold": OUTER_FOLD, "cnn_pred": p_ho[:, PATCH_PX // 2, PATCH_PX // 2]})
oof.to_csv(out(f"oof_fold{OUTER_FOLD}.csv"), index=False)
mlflow.log_artifact(out(f"oof_fold{OUTER_FOLD}.csv"))

sel_m = dense_metrics(h_map_va, predict_map_m(model, X_va), mask_va)
pd.DataFrame({f"val interna (fold {INNER_VAL_FOLD}, seleção)": sel_m, f"hold-out (fold {OUTER_FOLD})": ho_m}).T.round(3)

## 12. Seleção de patches (validação interna)
8 patches reais: 4 entre 15 e 30 m e 4 acima de 30 m (altura escalar do ponto), com RGB e LiDAR 23×23.

In [ ]:
import matplotlib.pyplot as plt

rng_vis = np.random.default_rng(SEED)
h_va = va["h"]
idx_15_30 = np.flatnonzero((h_va >= 15.0) & (h_va < 30.0))
idx_30p = np.flatnonzero(h_va >= 30.0)
cand = np.concatenate([rng_vis.choice(idx_15_30, size=min(4, len(idx_15_30)), replace=False),
                       rng_vis.choice(idx_30p, size=min(4, len(idx_30p)), replace=False)])
cand = cand[np.argsort(h_va[cand])]
gallery_sids = [va["sid"][i] for i in cand]

full33 = load_dense(TRAIN_DIR, TRAIN_PATTERN, S, sid_whitelist=set(gallery_sids))        # patches 33x33 completos (só validação)
pos = {s: i for i, s in enumerate(full33["sid"])}
half = PATCH_PX // 2

def rgb_of(x_raw):
    rgb = x_raw[..., [4, 3, 2]]
    p2, p98 = np.percentile(rgb, (2, 98))
    return np.clip((rgb - p2) / (p98 - p2 + 1e-6), 0.0, 1.0)

fig, axes = plt.subplots(2, len(cand), figsize=(2.9 * len(cand), 6))
gallery = []
for col, i in enumerate(cand):
    sid, k = va["sid"][i], pos[va["sid"][i]]
    lidar = full33["h_map"][k][half:S - half, half:S - half]
    axes[0, col].imshow(rgb_of(full33["x"][k]), interpolation="nearest")
    axes[0, col].add_patch(plt.Rectangle((half - 0.5, half - 0.5), S - 2 * half, S - 2 * half, edgecolor="yellow", facecolor="none", lw=1.2, ls="--"))
    axes[0, col].set_title(f"{sid}\n{va['city'][i]} | h: {h_va[i]:.1f} m", fontsize=8)
    axes[0, col].axis("off")
    axes[1, col].imshow(lidar, cmap="viridis", vmin=0, vmax=max(35.0, lidar.max()), interpolation="nearest")
    axes[1, col].set_title(f"LiDAR 23x23 | max {lidar.max():.1f} m", fontsize=8)
    axes[1, col].axis("off")
    gallery.append({"sample_id": sid, "cidade": va["city"][i], "h_ponto_m": round(float(h_va[i]), 2), "lidar_max_m": round(float(lidar.max()), 2)})
plt.tight_layout()
fig.savefig(out("figures/patch_gallery.png"), dpi=150, bbox_inches="tight")
plt.show()
mlflow.log_artifact(out("figures/patch_gallery.png"))
pd.DataFrame(gallery)

## 13. Mapas de altura (janela deslizante 11×11 → 23×23 centrais), modelos vs LiDAR
Cada janela prevê 11×11; as previsões sobrepostas são promediadas. Compara a baseline (U-Net compacta) com o modelo selecionado.

In [ ]:
CHOSEN_SID = "salvador_000402" #gallery_sids[-1]                                         # troque por qualquer sample_id da galeria acima
out_sz = S - 2 * half

def reconstruct_map(m, x56):
    windows = np.stack([x56[r:r + PATCH_PX, c:c + PATCH_PX] for r in range(out_sz) for c in range(out_sz)], axis=0)
    pw = predict_map_m(m, windows).reshape(out_sz, out_sz, PATCH_PX, PATCH_PX)
    acc, cnt = np.zeros((S, S)), np.zeros((S, S))
    for r in range(out_sz):
        for c in range(out_sz):
            acc[r:r + PATCH_PX, c:c + PATCH_PX] += pw[r, c]
            cnt[r:r + PATCH_PX, c:c + PATCH_PX] += 1.0
    return (acc / np.maximum(cnt, 1.0))[half:S - half, half:S - half].astype("float32")

base_path = out("models_arch/r1_u32_c2.keras")
vis_models = {"baseline (u32_c2)": base_path, f"selecionado ({best_row['key']})": BEST_PATH}
vis_models = {k: v for k, v in vis_models.items() if os.path.exists(v)}

def maps_for(sid):
    k = pos[sid] if sid in pos else None
    assert k is not None, f"{sid} não está na galeria: ajuste CHOSEN_SID"
    x56 = prep(full33["x"][k:k + 1], MEAN, STD)[0]
    lidar = full33["h_map"][k][half:S - half, half:S - half]
    preds = {}
    for name, path in vis_models.items():
        free_gpu()
        preds[name] = reconstruct_map(keras.models.load_model(path, compile=False), x56)
    return x56, lidar, preds

x56, lidar_gt, pred_maps = maps_for(CHOSEN_SID)
vmax_h = max(float(lidar_gt.max()), *[float(p.max()) for p in pred_maps.values()])

fig, axes = plt.subplots(2, 2 + len(pred_maps), figsize=(3.6 * (2 + len(pred_maps)), 7.2))
axes[0, 0].imshow(rgb_of(full33["x"][pos[CHOSEN_SID]])[half:S - half, half:S - half], interpolation="nearest")
axes[0, 0].set_title(f"Sentinel-2 RGB\n({CHOSEN_SID})", fontsize=9.5)
im_gt = axes[0, 1].imshow(lidar_gt, cmap="viridis", vmin=0, vmax=vmax_h, interpolation="nearest")
axes[0, 1].set_title(f"LiDAR (alvo)\nmax {lidar_gt.max():.1f} m", fontsize=9.5)
axes[1, 0].axis("off"); axes[1, 1].axis("off")
rows_tbl = []
for j, (name, pm) in enumerate(pred_maps.items(), start=2):
    err = np.abs(pm - lidar_gt)
    axes[0, j].imshow(pm, cmap="viridis", vmin=0, vmax=vmax_h, interpolation="nearest")
    axes[0, j].set_title(f"{name}\nmax {pm.max():.1f} m", fontsize=9)
    im_err = axes[1, j].imshow(err, cmap="magma", vmin=0, vmax=20.0, interpolation="nearest")
    axes[1, j].set_title(f"MAE {err.mean():.1f} m | RMSE {np.sqrt((err ** 2).mean()):.1f} m", fontsize=8.8)
    bl = lidar_gt > 0
    rows_tbl.append({"Modelo": name, "MAE_23x23": round(float(err.mean()), 2), "RMSE_23x23": round(float(np.sqrt((err ** 2).mean())), 2),
                     "MAE_prédios": round(float(err[bl].mean()), 2) if bl.any() else np.nan,
                     "RMSE_prédios": round(float(np.sqrt((err[bl] ** 2).mean())), 2) if bl.any() else np.nan,
                     "Max_pred": round(float(pm.max()), 2), "Max_LiDAR": round(float(lidar_gt.max()), 2)})
for ax in axes.ravel():
    ax.axis("off")
fig.colorbar(im_gt, ax=axes[0, :], shrink=0.75, label="Altura (m)")
fig.colorbar(im_err, ax=axes[1, 2:], shrink=0.75, label="Erro absoluto (m)")
plt.suptitle(f"Reconstrução espacial de altura (23x23 px = 230x230 m) – {CHOSEN_SID}", fontsize=12, y=1.02)
fig.savefig(out(f"figures/map_{CHOSEN_SID}.png"), dpi=150, bbox_inches="tight")
plt.show()
mlflow.log_artifact(out(f"figures/map_{CHOSEN_SID}.png"))
pd.DataFrame(rows_tbl)

In [ ]:
# modelo selecionado em todos os patches da galeria (linha 1: LiDAR | linha 2: predição)
m_best = keras.models.load_model(BEST_PATH, compile=False)
fig, axes = plt.subplots(2, len(gallery_sids), figsize=(2.9 * len(gallery_sids), 6))
for col, sid in enumerate(gallery_sids):
    k = pos[sid]
    lidar = full33["h_map"][k][half:S - half, half:S - half]
    pm = reconstruct_map(m_best, prep(full33["x"][k:k + 1], MEAN, STD)[0])
    vm = max(35.0, float(lidar.max()), float(pm.max()))
    axes[0, col].imshow(lidar, cmap="viridis", vmin=0, vmax=vm, interpolation="nearest")
    axes[0, col].set_title(f"{sid}\nLiDAR max {lidar.max():.1f} m", fontsize=8)
    axes[1, col].imshow(pm, cmap="viridis", vmin=0, vmax=vm, interpolation="nearest")
    axes[1, col].set_title(f"pred max {pm.max():.1f} m | MAE {np.abs(pm - lidar).mean():.1f} m", fontsize=8)
for ax in axes.ravel():
    ax.axis("off")
plt.tight_layout()
fig.savefig(out("figures/gallery_predictions.png"), dpi=150, bbox_inches="tight")
plt.show()
mlflow.log_artifact(out("figures/gallery_predictions.png"))
free_gpu()

## 14. Artefatos e encerramento do run

In [ ]:
np.savez(out("norm_stats.npz"), mean=MEAN, std=STD, patch_px=PATCH_PX)
json.dump({**CONFIG, "best_key": best_row["key"], "best_spec": BEST_SPEC}, open(out("config.json"), "w"), indent=1, default=str)
shutil.copy(BEST_PATH, out("best_model.keras"))
for f in ("norm_stats.npz", "config.json", "best_model.keras"):
    mlflow.log_artifact(out(f))
if "ctx" in globals() and len(ctx):
    ctx.to_csv(out("context_results.csv"), index=False)
    mlflow.log_artifact(out("context_results.csv"))
mlflow.end_run()                                                      # FINISHED: não será retomado de novo
print("run encerrado:", PARENT_RUN_ID)

## 15. Dashboard do MLflow (opcional)

In [ ]:
import subprocess, sys
from IPython.display import IFrame

ui = subprocess.Popen([sys.executable, "-m", "mlflow", "ui", "--backend-store-uri", MLFLOW_URI, "--port", str(MLFLOW_PORT)])
IFrame(f"http://127.0.0.1:{MLFLOW_PORT}", width="100%", height=700)

## 16. Teste final (NÃO executar antes de congelar o modelo)
A célula abaixo está inteiramente comentada para não rodar por acidente. Descomente e execute **uma única vez**, depois do hold-out. O RMSE ponderado por frequência fica para quando a tabela do projeto clássico estiver disponível.

In [ ]:
# ---- TESTE FINAL: descomente tudo e execute uma única vez, com o modelo congelado (BEST_PATH) ----
# mlflow.start_run(run_id=PARENT_RUN_ID)                                  # reabre o run pai já encerrado
# test = load_dense(TEST_DIR, TEST_PATTERN, PATCH_PX, only_split=None)
# X_te = prep(test["x"], MEAN, STD)                                       # usa MEAN/STD do treino real
# test.pop("x")
# free_gpu()
# model_final = keras.models.load_model(BEST_PATH, compile=False)
# p_te = predict_map_m(model_final, X_te)
#
# rows = {"test (todos)": dense_metrics(test["h_map"], p_te, test["mask_px"])}
# csv_files = glob.glob(DIST_GLOB)
# if csv_files:
#     dist = pd.concat([pd.read_csv(f) for f in csv_files])
#     clean = np.isin(test["sid"], dist.sample_id[dist.cheb_to_train_m >= PATCH_PX * 10].values)
#     rows["test (clean)"] = dense_metrics(test["h_map"][clean], p_te[clean], test["mask_px"][clean])
# else:
#     print(f"ATENÇÃO: nenhum CSV de distâncias em {DIST_GLOB}; teste 'clean' não calculado")
#
# for name, r in rows.items():
#     key = name.replace("test (", "test_").replace(")", "")
#     mlflow.log_metrics({f"{key}_{k}": float(v) for k, v in r.items()})
# mlflow.end_run()
# pd.DataFrame(rows).T.round(3)